In [1]:
import os
os.chdir('../')
os.getcwd()

'/home/minh_khai/skin_disease/skin-disease'

In [2]:
from abc import ABC, abstractmethod

class BaseIngestionAdapter(ABC):
    @abstractmethod
    def fetch(self, dst: str) -> None:
        pass

In [3]:
import os, sys, subprocess
from dotenv import load_dotenv

from core.logger import logger
from core.exception import CustomException

class KaggleIngestionAdapter(BaseIngestionAdapter):
    def __init__(self, dataset: str):
        self.dataset = dataset  # format: "owner/dataset-name"
        
    def fetch(self, dst: str) -> None:
        try: 
            load_dotenv()
            
            os.environ["KAGGLE_USERNAME"] = os.getenv("KAGGLE_USERNAME", "")
            os.environ["KAGGLE_KEY"]      = os.getenv("KAGGLE_API_TOKEN", "")

            import kaggle
            kaggle.api.authenticate()
            
            os.makedirs(dst, exist_ok=True)
            logger.info(f"Downloading Kaggle dataset {self.dataset} -> {dst}")
            
            subprocess.run(
                ["kaggle", "datasets", "download", "-d", self.dataset, "-p", dst, "--unzip", "--force"],
                check=True
            )
        except Exception as e:
            raise CustomException(e, sys)

In [4]:
class IngestionAdapterFactory:
    _adapters = {
        "KAGGLE": KaggleIngestionAdapter,
    }
    
    @staticmethod
    def create_adapter(source_type: str, source: str):
        source_type = source_type.upper()
        adapter_class = IngestionAdapterFactory._adapters.get(source_type)
        
        if not adapter_class:
            raise ValueError(f"Ingestion type '{source_type}' is not supported.")
        return adapter_class(source)

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class IngestionConfig:
    root_dir: Path
    src_type: str
    source:   str
    name:     str

In [ ]:
import os
from itertools import chain
from typing import List

from dotenv import load_dotenv
load_dotenv()

from core.constants import *
from core import read_yaml, create_directories

class ConfigurationManager:
    def __init__(self, 
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
    
    def _parse_sources(self, env_key: str) -> list:
        raw = os.getenv(env_key, "")
        if not raw.strip():
            return []
        
        sources = []
        for entry in raw.split(","):
            parts = entry.strip().split(":")
            
            if len(parts) != 3:
                continue
            
            name, src_type, source = parts
            sources.append({
                "name": name, 
                "src_type": src_type, 
                "source": source
            })
        
        return sources

    def get_ingestion_configs(self) -> List[IngestionConfig]:
        config = self.config.ingestion_config
        create_directories([config.root_dir])

        # ----- DEFINING ALL SOURCES <<<<<<<<<<<<
        local_src  = self._parse_sources("LOCAL_SOURCES")
        kaggle_src = self._parse_sources("KAGGLE_SOURCES")
        ee_src     = self._parse_sources("EE_SOURCES")
        
        configs = []
        for s in chain(local_src, kaggle_src, ee_src):
            configs.append(IngestionConfig(
                root_dir     = config.root_dir,
                src_type     = s["src_type"],
                source       = s["source"],
                name         = s["name"],
            ))

        return configs

In [7]:
import os, sys
from core.logger import logger
from core.exception import CustomException

class Ingestion:
    def __init__(self, config: IngestionConfig):
        self.config  = config
        self.adapter = IngestionAdapterFactory.create_adapter(
            self.config.src_type, self.config.source
        )
    
    def fetch_data(self) -> None:
        try: 
            dst = os.path.join(self.config.root_dir, self.config.name)
            self.adapter.fetch(dst)
            logger.info(f"Data fetched at {dst}")
        except Exception as e:
            raise CustomException(e, sys)

In [8]:
try:
    cfg_manager     = ConfigurationManager()
    ingestion_cfgs  = cfg_manager.get_ingestion_configs()
    for cfg in ingestion_cfgs:
        Ingestion(config=cfg).fetch_data()
except Exception as e:
    raise CustomException(e, sys)

[2026-06-15 09:48:32,194: INFO: __init__: yaml file: config/config.yaml loaded successfully]
[2026-06-15 09:48:32,196: INFO: __init__: yaml file: params.yaml loaded successfully]
[2026-06-15 09:48:32,198: INFO: __init__: created directory at: artifacts]
[2026-06-15 09:48:32,200: INFO: __init__: created directory at: artifacts/ingestion]
[2026-06-15 09:48:34,154: INFO: 3684487777: Downloading Kaggle dataset ismailpromus/skin-diseases-image-dataset -> artifacts/ingestion/MEKONG_SALINITY]
Dataset URL: https://www.kaggle.com/datasets/ismailpromus/skin-diseases-image-dataset
License(s): copyright-authors


100%|██████████| 5.19G/5.19G [04:36<00:00, 20.1MB/s]  



[2026-06-15 09:53:46,441: INFO: 214719300: Data fetched at artifacts/ingestion/MEKONG_SALINITY]
